In [1]:
import pandas as pd
import re
from collections import Counter
import copy
import time
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
import torch
from torch.utils.data import Dataset, DataLoader

In [2]:
amazon_labelled = pd.read_csv('sentiment labelled sentences\\amazon_cells_labelled.txt', sep='\t', names=['text', 'label'])
print(amazon_labelled.shape)
print(amazon_labelled.head())

(1000, 2)
                                                text  label
0  So there is no way for me to plug it in here i...      0
1                        Good case, Excellent value.      1
2                             Great for the jawbone.      1
3  Tied to charger for conversations lasting more...      0
4                                  The mic is great.      1


In [3]:
amazon_labelled['source'] = 'amazon'
print(amazon_labelled.head())

                                                text  label  source
0  So there is no way for me to plug it in here i...      0  amazon
1                        Good case, Excellent value.      1  amazon
2                             Great for the jawbone.      1  amazon
3  Tied to charger for conversations lasting more...      0  amazon
4                                  The mic is great.      1  amazon


In [4]:
imdb_labelled = pd.read_csv('sentiment labelled sentences\\imdb_labelled.txt', sep='\t', names=['text', 'label'], quoting=3)
print(imdb_labelled.shape)
print(imdb_labelled.head())
print(imdb_labelled.label.value_counts())

(1000, 2)
                                                text  label
0  A very, very, very slow-moving, aimless movie ...      0
1  Not sure who was more lost - the flat characte...      0
2  Attempting artiness with black & white and cle...      0
3       Very little music or anything to speak of.        0
4  The best scene in the movie was when Gerardo i...      1
label
0    500
1    500
Name: count, dtype: int64


In [5]:
imdb_labelled['source'] = 'imdb'
print(imdb_labelled.head())

                                                text  label source
0  A very, very, very slow-moving, aimless movie ...      0   imdb
1  Not sure who was more lost - the flat characte...      0   imdb
2  Attempting artiness with black & white and cle...      0   imdb
3       Very little music or anything to speak of.        0   imdb
4  The best scene in the movie was when Gerardo i...      1   imdb


In [6]:
yelp_labelled = pd.read_csv('sentiment labelled sentences\\yelp_labelled.txt', sep='\t', names=['text', 'label'])
print(yelp_labelled.shape)
print(yelp_labelled.head())

(1000, 2)
                                                text  label
0                           Wow... Loved this place.      1
1                                 Crust is not good.      0
2          Not tasty and the texture was just nasty.      0
3  Stopped by during the late May bank holiday of...      1
4  The selection on the menu was great and so wer...      1


In [7]:
yelp_labelled['source'] = 'yelp'
print(yelp_labelled.head())

                                                text  label source
0                           Wow... Loved this place.      1   yelp
1                                 Crust is not good.      0   yelp
2          Not tasty and the texture was just nasty.      0   yelp
3  Stopped by during the late May bank holiday of...      1   yelp
4  The selection on the menu was great and so wer...      1   yelp


In [8]:
data = pd.concat([amazon_labelled, imdb_labelled, yelp_labelled], ignore_index=True)

In [9]:
print('shape', data.shape)
print('count of NAN:\n', data.isna().sum())
print('distribution\n', data.groupby(['source', 'label']).size())

shape (3000, 3)
count of NAN:
 text      0
label     0
source    0
dtype: int64
distribution
 source  label
amazon  0        500
        1        500
imdb    0        500
        1        500
yelp    0        500
        1        500
dtype: int64


In [10]:
duplicates_text = data.duplicated(subset='text')
duplicates_all = data.duplicated(subset=['text', 'label'])
text_indexes = list(duplicates_text[duplicates_text == True].index)
all_indexes = list(duplicates_all[duplicates_all == True].index)

print(text_indexes == all_indexes)

True


In [11]:
data = data.drop_duplicates(subset=['text'], ignore_index=True)

print('shape', data.shape)
print(f'distribution\n{data.groupby(['source', 'label']).size()}')

shape (2983, 3)
distribution
source  label
amazon  0        497
        1        493
imdb    0        499
        1        498
yelp    0        497
        1        499
dtype: int64


In [12]:
stratify_tt = data.source + '_' + data.label.astype('str')
X_trainval, X_test, y_trainval, y_test = train_test_split(
    data.text, data.label,
    test_size=0.15,
    random_state=42,
    stratify=stratify_tt
)


stratify_tv = data.loc[X_trainval.index].source + '_' + data.loc[X_trainval.index].label.astype('str')
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.18,
    random_state=42,
    stratify=stratify_tv
)

print(f'X_train shape: {X_train.shape}\nX_val shape: {X_val.shape}\nX_test shape: {X_test.shape}')
print(data.loc[X_test.index].groupby(['source', 'label']).size())
print(data.loc[X_train.index].groupby(['source', 'label']).size())
print(data.loc[X_val.index].groupby(['source', 'label']).size())

X_train shape: (2078,)
X_val shape: (457,)
X_test shape: (448,)
source  label
amazon  0        75
        1        74
imdb    0        75
        1        75
yelp    0        74
        1        75
dtype: int64
source  label
amazon  0        346
        1        343
imdb    0        348
        1        347
yelp    0        347
        1        347
dtype: int64
source  label
amazon  0        76
        1        76
imdb    0        76
        1        76
yelp    0        76
        1        77
dtype: int64


Разделение сохранило состав данных по источникам и классам.

In [13]:
def tokenize(text: str) -> list:
    text = text.lower()
    text = text.replace('’', "'")
    tokens = re.findall(r'[a-zа-яё]+(?:[\'][а-яёa-z]+)*|\.|\,|\?|\!', text)

    return tokens

In [14]:
train_tokens = X_train.apply(tokenize)
print("все отзывы токенизировались" if len(train_tokens) == X_train.shape[0] else 'не все отзывы токенизировались')
print('индексы сохранились' if train_tokens.index.equals(X_train.index) else 'индексы не сохранились')
print(sum(1 for l in train_tokens if len(l) == 0))
print(train_tokens)

все отзывы токенизировались
индексы сохранились
0
2039                 [their, chow, mein, is, so, good, !]
2482                      [and, then, tragedy, struck, .]
1136    [the, fish, is, badly, made, and, some, of, it...
2169     [i, dressed, up, to, be, treated, so, rudely, !]
1650    [however, paul, schrader, has, indeed, made, a...
                              ...                        
1868    [i, was, bored, throughout, the, whole, damn, ...
2556                        [i, consider, this, theft, .]
1804    [you, will, leave, the, theater, wanting, to, ...
2264    [the, goat, taco, didn't, skimp, on, the, meat...
1338                                [it's, just, lame, .]
Name: text, Length: 2078, dtype: object


In [15]:
def get_tokens_frequency(list_of_tokens):
    total = [l[w] for l in list_of_tokens for w in range(len(l))]
    tokens_freq = Counter(t for t in total)

    return tokens_freq


train_tokens_freq = get_tokens_frequency(train_tokens)
ten_most_common = train_tokens_freq.most_common(10)
print(*ten_most_common, sep='\n')

('.', 2125)
('the', 1335)
(',', 913)
('and', 811)
('i', 637)
('a', 613)
('is', 509)
('to', 479)
('it', 478)
('this', 467)


**Сравнение двух вариантов без построения словаря**: сколько обычных токенов останется при min_freq=1 и при min_freq=2

In [16]:
total = [l[w] for l in train_tokens for w in range(len(l))]
l1, l2 = [], []
for k, v in train_tokens_freq.items():
    if v < 1:
        continue
    l1.append(v)

for k, v in train_tokens_freq.items():
    if v < 2:
        continue
    l2.append(v)

skip_1 = len(total) - sum(l1)
skip_2 = len(total) - sum(l2)

print(f"Всего {len(total)} слов")
print(f'При min_freq=1 мы пропустим {skip_1} токена(-ов), останется {len(l1)} уникальных токена(-ов).')
print(f'При min_freq=2 пропустим {skip_2} токена(-ов), останется {len(l2)} уникальных токена(-ов).')
print(f'При min_freq=1 будет заменено {skip_1 / len(total) * 100:0.1f}% слов.')
print(f'При min_freq=2 будет заменено {skip_2 / len(total) * 100:0.1f}% слов.')

Всего 28235 слов
При min_freq=1 мы пропустим 0 токена(-ов), останется 4193 уникальных токена(-ов).
При min_freq=2 пропустим 2426 токена(-ов), останется 1767 уникальных токена(-ов).
При min_freq=1 будет заменено 0.0% слов.
При min_freq=2 будет заменено 8.6% слов.


In [17]:
def make_dict(tokens_freq: dict, min_freq = 1):
    dictionary = {}
    dictionary.setdefault('<PAD>', 0)
    dictionary.setdefault('<UNK>', 1)
    i = 2
    for k, v in tokens_freq.items():
        if v < min_freq:
            continue
        dictionary.setdefault(k, i)
        i += 1

    return dictionary


dictionary = make_dict(train_tokens_freq)
print(len(dictionary))
print('Is max index == (len(dictionary) - 1)?', max(dictionary.values()) == (len(dictionary) - 1))
for k, v in dictionary.items():
    print(f'({k}, {v})')
    if v == 10:
        break

4195
Is max index == (len(dictionary) - 1)? True
(<PAD>, 0)
(<UNK>, 1)
(their, 2)
(chow, 3)
(mein, 4)
(is, 5)
(so, 6)
(good, 7)
(!, 8)
(and, 9)
(then, 10)


In [18]:
def tokens_to_ids(tokens: list, dictionary) -> list:
    ids = []
    for token in tokens:
        ind = dictionary.get(token, dictionary['<UNK>'])
        ids.append(ind)
    return ids

In [19]:
def check_encoder(encode_tokens, dictionary):
    dictionary_before = dictionary.copy()

    known = next(t for t in dictionary if t not in ("<PAD>", "<UNK>"))
    unknown = "__несуществующий_токен__"
    while unknown in dictionary:
        unknown += "_"

    tokens = [known, unknown, known]
    tokens_before = tokens.copy()

    assert encode_tokens([], dictionary) == [], \
        "Пустой список должен давать []"

    result = encode_tokens(tokens, dictionary)

    assert isinstance(result, list), "Результат должен быть списком"
    assert result == [
        dictionary[known],
        dictionary["<UNK>"],
        dictionary[known],
    ], "Нарушены индексы, порядок или обработка неизвестного токена"

    assert tokens == tokens_before, "Функция изменила входной список"
    assert dictionary == dictionary_before, "Функция изменила словарь"

    print("Все проверки пройдены")


check_encoder(tokens_to_ids, dictionary)

Все проверки пройдены


In [20]:
train_ids = []
for l in train_tokens:
    train_ids.append(tokens_to_ids(l, dictionary=dictionary))

In [21]:
val_tokens = X_val.apply(tokenize)
val_ids = []
for l in val_tokens:
    val_ids.append(tokens_to_ids(l, dictionary))

In [22]:
pairs = [([4, 7], 0), ([], 1), ([9], 0)]

def collate_fn(pairs) -> (torch.Tensor, torch.Tensor, torch.Tensor):
    seqs = []
    labels = []
    for pair in pairs:
        seq = pair[0] if len(pair[0]) != 0 else [1]
        seqs.append(torch.tensor(seq, dtype=torch.long))
        labels.append(pair[1])

    PAD_id = dictionary['<PAD>']
    seqs_pad = torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=PAD_id)

    mask = seqs_pad != PAD_id

    return seqs_pad, mask, torch.tensor(labels, dtype=torch.long)

seqs, mask, labels = collate_fn(pairs)
print(seqs, mask, labels, sep='\n')
print(f'shape of sequences is {seqs.shape}')
print(f'shape of mask is {mask.shape}')
print(f'shape of labels is {labels.shape}')
print('seqs.dtype is', seqs.dtype, 'mask.dtype is', mask.dtype, 'labels.dtype is', labels.dtype)

tensor([[4, 7],
        [1, 0],
        [9, 0]])
tensor([[ True,  True],
        [ True, False],
        [ True, False]])
tensor([0, 1, 0])
shape of sequences is torch.Size([3, 2])
shape of mask is torch.Size([3, 2])
shape of labels is torch.Size([3])
seqs.dtype is torch.int64 mask.dtype is torch.bool labels.dtype is torch.int64


In [23]:
class MyDataset(Dataset):
    def __init__(self, data, labels):
        assert len(data) == len(labels), 'Количество примеров и меток не совпадает'
        self.data = data
        self.labels = labels
    def __len__(self):
        return len(self.data)
    def __getitem__(self, ind):
        return self.data[ind], self.labels.iloc[ind]


dataset = MyDataset(train_ids, y_train)
print(f"Размер датасета: {len(dataset)}")
print(f"Первый элемент: {dataset[0]}")

Размер датасета: 2078
Первый элемент: ([2, 3, 4, 5, 6, 7, 8], np.int64(1))


In [24]:
train_dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn, generator=torch.Generator().manual_seed(123))

iter_dataloader = iter(train_dataloader)
seqs, mask, labels = next(iter_dataloader)
print('seqs shape is', seqs.shape)
print('mask shape is', mask.shape)
print('labels shape is', labels.shape)

seqs shape is torch.Size([32, 21])
mask shape is torch.Size([32, 21])
labels shape is torch.Size([32])


In [25]:
embedding = torch.nn.Embedding(len(dictionary), 32, padding_idx=dictionary['<PAD>'])
vectors = embedding(seqs)
vectors.shape

torch.Size([32, 21, 32])

In [26]:
unsqeezed_mask = torch.unsqueeze(mask, dim=-1)
vector_sum = (unsqeezed_mask * vectors).sum(dim=1)
print('Сумма по токенам', vector_sum.shape)

l = mask.sum(dim=1, keepdim=True)
mask_mean_pooling = vector_sum / l
print("mask mean pooling", mask_mean_pooling.shape)

Сумма по токенам torch.Size([32, 32])
mask mean pooling torch.Size([32, 32])


In [27]:
class Model(torch.nn.Module):
    def __init__(self, dictionary):
        super().__init__()
        self.embedding = torch.nn.Embedding(len(dictionary), 32, padding_idx=dictionary['<PAD>'])
        self.linear = torch.nn.Linear(32, 2)

    def forward(self, seq, mask):
        x = self.embedding(seq)
        unsqeezed_mask = torch.unsqueeze(mask, dim=-1)
        vector_sum = (unsqeezed_mask * x).sum(dim=1)
        l = mask.sum(dim=1, keepdim=True)
        mask_mean_pooling = vector_sum / l
        logits = self.linear(mask_mean_pooling)
        return logits

model = Model(dictionary)
pred = model(seqs, mask)

loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(pred, labels)
print(loss, loss.shape)

tensor(0.7069, grad_fn=<NllLossBackward0>) torch.Size([])


In [28]:
model1 = Model(dictionary)
lr = 0.01
optimizer = torch.optim.Adam(model1.parameters(), lr=lr)
seqs, mask, labels = next(iter_dataloader)
model1.train()
for i in range(32):
    x = model1(seqs, mask)
    loss = loss_fn(x, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    accuracy = (x.argmax(dim=1) == labels).sum().item() / 32 * 100

    if i % 10 == 0:
        print('loss', loss.item())
        print('accuracy ', accuracy, '%', sep='')

loss 0.6841371655464172
accuracy 50.0%
loss 0.4170718789100647
accuracy 87.5%
loss 0.1734882891178131
accuracy 100.0%
loss 0.04481479525566101
accuracy 100.0%


In [29]:
def train_model(model, dataloader, loss_fn, optimizer):
    model.train()
    total_loss = 0

    for s, m, l in dataloader:
        x = model(s, m)
        loss = loss_fn(x, l)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * s.shape[0]

    return total_loss / len(dataloader.dataset)

In [30]:
val_dataset = MyDataset(val_ids, y_val)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [31]:
def evaluate_model(model, dataloader, loss_fn):
    model.eval()

    total_loss, total_accuracy = 0, 0
    with torch.no_grad():
        for s, m, l in dataloader:
            x = model(s, m)
            loss = loss_fn(x, l)
            total_loss += loss.item() * s.shape[0]
            total_accuracy += (x.argmax(dim=1) == l).sum().item()

    avg_loss = total_loss / len(dataloader.dataset)
    avg_accuracy = total_accuracy / len(dataloader.dataset) * 100
    return avg_loss, avg_accuracy

In [32]:
val_loss, val_accuracy = evaluate_model(model, val_dataloader, loss_fn)
print('val_loss ', val_loss, '\nval_accuracy ', val_accuracy, sep='')

val_loss 0.7158621557431878
val_accuracy 48.57768052516411


In [33]:
torch.manual_seed(123)
model = Model(dictionary)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
epoch = 10
min_loss = float('inf')
best_model = None
best_epoch = 0
epoch_accuracy = 0


train_dataloader.generator.manual_seed(123)
for e in range(epoch):
    train_model(model, train_dataloader, loss_fn, optimizer)
    loss, accuracy = evaluate_model(model, val_dataloader, loss_fn)

    if loss < min_loss:
        min_loss = loss
        best_model = copy.deepcopy(model.state_dict())
        best_epoch = e + 1
        epoch_accuracy = accuracy

model.load_state_dict(best_model)
assert evaluate_model(model, val_dataloader, loss_fn) == (min_loss, epoch_accuracy), 'Лучшая модель не сохранена'

print(f'Best epoch is {best_epoch} with min loss {min_loss} and accuracy {epoch_accuracy}')

Best epoch is 3 with min loss 0.4274111436415032 and accuracy 79.86870897155362


**Проверка модели на устойчивость**

In [34]:
seed = 2026
torch.manual_seed(seed)
model = Model(dictionary)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
min_loss = float('inf')
epoch_accuracy = 0
best_epoch = 0

train_dataloader.generator.manual_seed(123)
for e in range(epoch):
    train_model(model, train_dataloader, loss_fn, optimizer)
    loss, accuracy = evaluate_model(model, val_dataloader, loss_fn)

    if loss < min_loss:
        min_loss = loss
        epoch_accuracy = accuracy
        best_epoch = e + 1

print(f'seed = {seed}\nBest epoch is {best_epoch} with min loss {min_loss} and accuracy {epoch_accuracy}')

seed = 2026
Best epoch is 4 with min loss 0.4251581003942427 and accuracy 84.9015317286652


In [35]:
seed = 42
torch.manual_seed(seed)
model = Model(dictionary)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
min_loss = float('inf')
epoch_accuracy = 0
best_epoch = 0

train_dataloader.generator.manual_seed(123)
for e in range(epoch):
    train_model(model, train_dataloader, loss_fn, optimizer)
    loss, accuracy = evaluate_model(model, val_dataloader, loss_fn)

    if loss < min_loss:
        min_loss = loss
        epoch_accuracy = accuracy
        best_epoch = e + 1

print(f'seed = {seed}\nBest epoch is {best_epoch} with min loss {min_loss} and accuracy {epoch_accuracy}')

seed = 42
Best epoch is 3 with min loss 0.41591741020752515 and accuracy 81.1816192560175


По loss результаты близки, но правильность выбора заметно зависит от инициализации.

##### TF-IDF and logistic regression

In [36]:
vectorizer = TfidfVectorizer()
train_tfidf_matrix = vectorizer.fit_transform(X_train)
val_tfidf_matrix = vectorizer.transform(X_val)
print('train matrix shape', train_tfidf_matrix.shape)
print('val matrix shape', val_tfidf_matrix.shape)

train matrix shape (2078, 4187)
val matrix shape (457, 4187)


In [37]:
log_regressor = LogisticRegression()
log_regressor.fit(train_tfidf_matrix, y_train)

y_pred_proba = log_regressor.predict_proba(val_tfidf_matrix)
y_pred = (y_pred_proba[:, 1] >= 0.5).astype(int)
accuracy = accuracy_score(y_val, y_pred)
logloss = log_loss(y_val, y_pred_proba)
print(f'accuracy on val {accuracy}')
print(f'log loss on val {logloss}')

accuracy on val 0.8271334792122538
log loss on val 0.4806967481508112


In [38]:
min_logloss = float('inf')
best_c = 0
c_accuracy = 0
c_logregressor = None

for c in [0.1, 1, 10]:
    log_regressor = LogisticRegression(C=c)
    log_regressor.fit(train_tfidf_matrix, y_train)

    y_pred_proba = log_regressor.predict_proba(val_tfidf_matrix)
    y_pred = (y_pred_proba[:, 1] >= 0.5).astype(int)
    accuracy = accuracy_score(y_val, y_pred)
    logloss = log_loss(y_val, y_pred_proba)

    if min_logloss > logloss:
        min_logloss = logloss
        best_c = c
        c_accuracy = accuracy
        c_logregressor = log_regressor


print(f'C = {best_c} with min log loss on val {min_logloss} and accuracy on val {c_accuracy}')

C = 10 with min log loss on val 0.37841884672265774 and accuracy on val 0.849015317286652


**Вывод**

Best epoch is 3 with min loss 0.4274111436415032 and accuracy 79.86870897155362

vs

C = 10 with min log loss on val 0.37841884672265774 and accuracy on val 0.849015317286652


TF-IDF + Logistic Regression с C=10 пока лучше запуска нейросети по обеим validation-метрикам в среднем, учитывая, что результат модели зависит от запуска.

**Анализ ошибок двух моделей после обучения**

In [39]:
model.eval()
nn_pred = []
with torch.no_grad():
    for s, m, l in val_dataloader:
        nn_pred.extend(model(s, m).argmax(dim=1).tolist())

nn_pred = pd.Series(nn_pred, index=X_val.index)

tabl = pd.DataFrame()
tabl['review'] = X_val
tabl['label'] = y_val
tabl['nn_pred'] = nn_pred
tabl['lr_pred'] = c_logregressor.predict(val_tfidf_matrix)

print(tabl.head(10))

                                                 review  label  nn_pred  \
144   The phone takes FOREVER to charge like 2 to 5 ...      0        0   
1027              Buy it, play it, enjoy it, love it.        1        1   
2848  They have a plethora of salads and sandwiches,...      1        1   
841       I bought two of them and neither will charge.      0        1   
2530   I won't try going back there even if it's empty.      0        0   
314   It's uncomfortable and the sound quality is qu...      0        0   
124                                  Gets the job done.      1        1   
1337  " But "Storm Trooper" is not even bad enough t...      0        0   
919                  As I said above....PRETTY USELESS!      0        0   
267                                  It's a great item.      1        1   

      lr_pred  
144         0  
1027        1  
2848        1  
841         0  
2530        0  
314         0  
124         1  
1337        0  
919         0  
267         1 

In [40]:
print('Нейросеть ошиблась, логистическая регрессия - нет')
nn_mist_lr_right = tabl[(tabl['nn_pred'] != tabl['label']) & (tabl['lr_pred'] == tabl['label'])]
print(nn_mist_lr_right.head(5))


lr_mist_nn_right = tabl[(tabl['lr_pred'] != tabl['label']) & (tabl['nn_pred'] == tabl['label'])]
print('\nЛогистическая регрессия ошиблась, нейросеть - нет')
print(lr_mist_nn_right.head(5))

Нейросеть ошиблась, логистическая регрессия - нет
                                                 review  label  nn_pred  \
841       I bought two of them and neither will charge.      0        1   
15    Highly recommend for any one who has a blue to...      1        0   
1439                            Wow, what a bad film.        0        1   
1246                                  That was funny.        1        0   
2888  Also, I feel like the chips are bought, not ma...      0        1   

      lr_pred  
841         0  
15          1  
1439        0  
1246        1  
2888        0  

Логистическая регрессия ошиблась, нейросеть - нет
                                                 review  label  nn_pred  \
2624  I was proven dead wrong by this sushi bar, not...      1        1   
1994  The potatoes were like rubber and you could te...      0        0   
1736  I'm a big fan of this series mostly due to Ann...      1        1   
1886  Director Neil LaBute uses brutal violence to s

**гипотезы об ошибках**

- Редкие слова плохо выучены нейросетью.
- "wow" сдвигает предсказание к положительному классу.
- Неоднозначность, отрицание, переносный смысл ухудшают предсказания моделям.

In [41]:
assert 'neither' in dictionary.keys(), '"neither" is <UNK>'
print(f'frequenct of "neither" - {train_tokens_freq['neither']}')

assert 'funny' in dictionary.keys(), '"funny" is <UNK>'
print(f'frequenct of "funny" - {train_tokens_freq['funny']}')

assert 'wow' in dictionary.keys(), '"wow" is <UNK>'
print(f'frequenct of "wow" - {train_tokens_freq['wow']}')

assert 'bad' in dictionary.keys(), '"bad" is <UNK>'
print(f'frequenct of "bad" - {train_tokens_freq['bad']}')

assert 'recommend' in dictionary.keys(), '"recommend" is <UNK>'
print(f'frequenct of "recommend" - {train_tokens_freq['recommend']}')

frequenct of "neither" - 2
frequenct of "funny" - 15
frequenct of "wow" - 4
frequenct of "bad" - 75
frequenct of "recommend" - 32


Редкость оказалась правдоподобным объяснением для neither, но причину ошибки это не доказало. Все токены входят в словарь.

In [42]:
r = ['Wow, what a bad film.', ]
r_tokens = tokenize(r[0])
r_ids = tokens_to_ids(r_tokens, dictionary)
r_seq, r_mask, _ = collate_fn([(r_ids, tabl.loc[1439]['label']),])

r2 = [', What a bad film.', ]
r2_tokens = tokenize(r2[0])
r2_ids = tokens_to_ids(r2_tokens, dictionary)
r2_seq, r2_mask, _ = collate_fn([(r2_ids, 0), ])

model.eval()
with torch.no_grad():
    y = model(r_seq, r_mask).softmax(dim=1)
    y2 = model(r2_seq, r2_mask).softmax(dim=1)

print('neg probabilities for "Wow, what a bad film." and ", What a bad film.", respectively', y[:, 0], y2[:, 0], sep='\n')

neg probabilities for "Wow, what a bad film." and ", What a bad film.", respectively
tensor([0.3674])
tensor([0.9780])


"Wow, what a bad film." -> вероятность отрицательного класса 36,74%
", what a bad film." -> 97,80%

Гипотеза подтвердилась для этого примера.

**Время исполнения**

Теперь сравнение времени обучения модели за 10 эпох и TF-IDF с логистической регрессией.

In [43]:
t_start = time.perf_counter()
vectorizer = TfidfVectorizer()
train_tfidf_matrix = vectorizer.fit_transform(X_train)
val_tfidf_matrix = vectorizer.transform(X_val)
tfidf_time = time.perf_counter() - t_start

t_start = time.perf_counter()
log_regressor = LogisticRegression(C=best_c)
log_regressor.fit(train_tfidf_matrix, y_train)
y_pred_proba = log_regressor.predict_proba(val_tfidf_matrix)
y_pred = (y_pred_proba[:, 1] >= 0.5).astype(int)
accuracy = accuracy_score(y_val, y_pred)
logloss = log_loss(y_val, y_pred_proba)
lr_time = time.perf_counter() - t_start

torch.manual_seed(123)
model = Model(dictionary)
train_dataloader.generator.manual_seed(123)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
min_loss = float('inf')
best_model = None
epoch_accuracy = 0
best_epoch = 0
t_start = time.perf_counter()
for e in range(epoch):
    train_model(model, train_dataloader, loss_fn, optimizer)
    loss, accuracy = evaluate_model(model, val_dataloader, loss_fn)
    if loss < min_loss:
        min_loss = loss
        best_model = copy.deepcopy(model.state_dict())
        best_epoch = e + 1
        epoch_accuracy = accuracy

nn_time = time.perf_counter() - t_start

print(f'TF-IDF: построение train-признаков и преобразование validation: {tfidf_time:.3f}')
print(f'Logistic Regression: обучение и оценка: {lr_time:.3f}')
print(f'TF-IDF + Logistic Regression: {tfidf_time + lr_time:.3f}')
print(f'нейросеть: 10 эпох, validation и сохранение лучших весов: {nn_time:.3f}')

TF-IDF: построение train-признаков и преобразование validation: 0.047
Logistic Regression: обучение и оценка: 0.027
TF-IDF + Logistic Regression: 0.074
нейросеть: 10 эпох, validation и сохранение лучших весов: 1.362


**Вывод**

В скорости TF-IDF + Logistic Regression тоже дало лучший результат

**Переход к тестовой выборке**

In [44]:
# Neural Network
torch.manual_seed(123)
final_model = Model(dictionary)
final_model.load_state_dict(best_model)

test_tokens = X_test.apply(tokenize)
test_ids = []
for l in test_tokens:
    test_ids.append(tokens_to_ids(l, dictionary))
test_dataset = MyDataset(test_ids, y_test)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
nn_loss, nn_accuracy = evaluate_model(final_model, test_dataloader, loss_fn)
print(f'Neural Network best weight for 10 epoch: loss - {nn_loss}, accuracy - {nn_accuracy}')


# Logistic Regression
test_tfidf_matrix = vectorizer.transform(X_test)
tfidf_lr_pred_proba = log_regressor.predict_proba(test_tfidf_matrix)
tfidf_lr_pred = (tfidf_lr_pred_proba[:, 1] >= 0.5).astype(int)
logloss = log_loss(y_test, tfidf_lr_pred_proba)
tfidf_lr_accuracy = accuracy_score(y_test, tfidf_lr_pred) * 100
print(f'TF-IDF + Logistic Regression: log loss - {logloss}, accuracy - {tfidf_lr_accuracy}')

Neural Network best weight for 10 epoch: loss - 0.4636523627809116, accuracy - 78.57142857142857
TF-IDF + Logistic Regression: log loss - 0.3882082292454797, accuracy - 81.25


Я бы выбрала TF-IDF с логистической регрессией: в нашем эксперименте оно показало лучшее качество на test и обучалось быстрее. Ограничение текущего представления в потери порядка слов, из-за которой модель плохо различает некоторые конструкции с отрицаниями.

**Сохранение конечного алгоритма**

In [ ]:
if not os.path.exists('algorithm'):
    os.mkdir('algorithm')
filepath = os.path.join('algorithm', 'algorithm.joblib')

algorithms = {'vectorizer': vectorizer, 'log_regressor': log_regressor}
joblib.dump(algorithms, filepath)